In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))

len(files)

3512

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()


nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 480


view_new = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=-30 * 3600) ### North pole view
#view_new = View(nx, ny, xc, yc, Rsun, 90, -90, 0, rsun_arc=-30 * 3600) ### South pole view

grid = view_new.grid()
mean,  coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0])

for i,file in enumerate(files[:]):
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    did = int(file.split('_')[-1].split('.')[0])
    date = file.split('/')[-1].split('_')[3]

    data = undistort(data, header, xd, yd)

    view = View.from_header(header)
    view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

    transform = (view_new.to_carrington(correct_mu=True) -
                 view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=0.15, stonyhurst=False))
    grid_, alpha = transform(grid)
    data = interp2d(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 4

    coverage += np.nan_to_num(n - coverage) / 10
    mean += np.nan_to_num((data - mean) * n / coverage / 10)

    if i % 10 == 0:
        out = mean.copy()
        out[coverage == 0] = np.nan

        show_data(out, view_new, title=date, label=r'$B_{los}$', to_file=f'temp/{i//10:04d}.png')

KeyboardInterrupt: 